In [ ]:
# !pip install datasets

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

In [ ]:

# NSMC (네이버 영화 리뷰 감성 분석) 데이터 활용 예시
# 실제 사용 시: https://github.com/e9t/nsmc 에서 다운로드
# 또는 datasets 라이브러리: pip install datasets
from datasets import load_dataset
nsmc = load_dataset(
    "csv",
    data_files={
        "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
    },
    sep="\t"
)
# DatasetDict({
#     train: Dataset({features: ['id', 'document', 'label'], num_rows: 150000}),
#     test:  Dataset({features: ['id', 'document', 'label'], num_rows: 50000})
# })
# print(nsmc)

class NSMCDataset(Dataset):
    """네이버 영화 리뷰 감성 분류 데이터셋"""
    
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data[idx]
        text = str(row['document']) if row['document'] else ""
        label = int(row['label'])
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 데이터로더 생성
MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NSMCDataset(nsmc['train'], tokenizer)
test_dataset  = NSMCDataset(nsmc['test'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4)

print(f"훈련: {len(train_dataset):,}개 / 테스트: {len(test_dataset):,}개")



In [ ]:
# Trainer를 사용하면 학습 루프를 직접 작성하지 않아도 됨
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import load_metric

# AutoModelForSequenceClassification: 분류 헤드가 내장된 모델
model_hf = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy="epoch",  # 에폭마다 평가
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    fp16=torch.cuda.is_available(),  # GPU 있으면 혼합 정밀도
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model_hf,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# 단 한 줄로 훈련 시작
trainer.train()
trainer.evaluate()


In [ ]:
from torch.optim import AdamW

# BERT 레이어는 낮은 lr, 분류기는 높은 lr
# → 사전학습 가중치를 보존하면서 분류기를 빠르게 학습

optimizer = AdamW([
    {'params': model.bert.parameters(),        'lr': 2e-5},  # BERT: 낮은 lr
    {'params': model.classifier.parameters(),  'lr': 1e-4},  # Classifier: 높은 lr
], weight_decay=0.01)


In [ ]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

# 스케줄러: warmup + linear decay
NUM_EPOCHS = 3
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = total_steps // 10  # 전체의 10%를 warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        
        # 그래디언트 클리핑 (BERT 훈련 시 중요)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

# 전체 훈련 루프
best_acc = 0.0
print(f"{'Epoch':>5} {'Train Loss':>10} {'Train Acc':>9} {'Val Loss':>8} {'Val Acc':>7}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, device)
    
    print(f"{epoch:>5} {train_loss:>10.4f} {train_acc:>9.4f} {val_loss:>8.4f} {val_acc:>7.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_bert_nsmc.pt")
        print(f"  ✓ 최고 모델 저장 (val_acc={val_acc:.4f})")
